In [10]:
import click
import sys
sys.path.append('C:/Users/inesm/projectos/tennis-predictor/src/data/')
from make_dataset import *
sys.path.append('C:/Users/inesm/projectos/tennis-predictor/src/features/')
from build_features import *
import pandas as pd
import seaborn as sns
input_filepath = 'C:/Users/inesm/projectos/tennis-predictor/data/interim/'
output_filepath = 'C:/Users/inesm/projectos/tennis-predictor/data/interim/'

In [2]:
dataset = load_data(input_filepath, file_name='dataset.pkl')
dataset

,Winner,Loser,WRank,LRank,B365W,B365L,Surface,match_id
1371,Johansson T.,Pretzsch A.,7,133,1.100,6.500,Grass,1371
1413,Morrison J.,Ilie A.,98,176,1.571,2.250,Grass,1413
1409,Lee H.T.,Larsson M.,117,142,2.625,1.444,Grass,1409
1407,Godwin N.,Gasquet R.,111,281,1.533,2.375,Grass,1407
1384,Youzhny M.,Santoro F.,67,32,1.800,1.909,Grass,1384
...,...,...,...,...,...,...,...,...
36335,Bautista Agut R.,Pouille L.,23,15,1.660,2.200,Hard,36335
36302,Anderson K.,Donaldson J.,8,59,1.300,3.500,Hard,36302
36361,Jarry N.,Zeballos H.,73,69,1.900,1.900,Clay,36361
36304,Del Potro J.M.,Anderson K.,9,8,1.500,2.620,Hard,36304


In [3]:
print(dataset.info())

<class 'pandas.core.frame.DataFrame'>
Index: 32375 entries, 1371 to 36362
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Winner    32375 non-null  object 
 1   Loser     32375 non-null  object 
 2   WRank     32375 non-null  int32  
 3   LRank     32375 non-null  int32  
 4   B365W     32375 non-null  float64
 5   B365L     32375 non-null  float64
 6   Surface   32375 non-null  object 
 7   match_id  32375 non-null  int64  
dtypes: float64(2), int32(2), int64(1), object(3)
memory usage: 2.0+ MB
None


In [5]:
# Add rankings of each player
dataset['RankP1'] = dataset.apply(calculate_p1_rank, axis=1)
dataset['RankP2'] = dataset.apply(calculate_p2_rank, axis=1)
classify_feature(dataset, 'RankP1', num_classes=10)
classify_feature(dataset, 'RankP2', num_classes=10)

# Add head-to-head
dataset = calculate_h2h(dataset)

# Add difference of rankings
dataset = calculate_rank_dif(dataset)
classify_feature(dataset, 'Rank_dif', num_classes=10)

# Add a binary column for each surface
dataset = OHE_surface(dataset)

### Add features for class model
# Add players odds
dataset = add_players_odds(dataset)
classify_feature(dataset, 'OddP1', num_classes=10)
classify_feature(dataset, 'OddP2', num_classes=10)

# Add difference of odds
dataset = calculate_odd_dif(dataset)
classify_feature(dataset, 'Odd_dif', num_classes=10)

# Add probability of P1 being the winner based on the odds 
dataset = add_prob_p1_wins(dataset)
classify_feature(dataset, 'Y_B365', num_classes=10)

df doesn't contain surface.


,Winner,Loser,WRank,LRank,B365W,B365L,match_id,RankP1,RankP2,RankP1_classified,...,Surface_Grass,Surface_Hard,OddP1,OddP2,OddP1_classified,OddP2_classified,Odd_dif,Odd_dif_classified,Y_B365,Y_B365_classified
1371,Johansson T.,Pretzsch A.,7,133,1.100,6.500,1371,7,133,1,...,1,0,1.100,6.500,0,8,-5.400,1,0.855263,8
1413,Morrison J.,Ilie A.,98,176,1.571,2.250,1413,98,176,9,...,1,0,1.571,2.250,7,2,-0.679,7,0.588851,2
1409,Lee H.T.,Larsson M.,117,142,2.625,1.444,1409,142,117,9,...,1,0,1.444,2.625,6,4,-1.181,5,0.645122,3
1407,Godwin N.,Gasquet R.,111,281,1.533,2.375,1407,111,281,9,...,1,0,1.533,2.375,7,3,-0.842,6,0.607728,3
1384,Youzhny M.,Santoro F.,67,32,1.800,1.909,1384,67,32,7,...,1,0,1.800,1.909,9,0,-0.109,9,0.514694,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36335,Bautista Agut R.,Pouille L.,23,15,1.660,2.200,36335,23,15,3,...,0,1,1.660,2.200,8,1,-0.540,8,0.569948,1
36302,Anderson K.,Donaldson J.,8,59,1.300,3.500,36302,8,59,1,...,0,1,1.300,3.500,3,6,-2.200,3,0.729167,6
36361,Jarry N.,Zeballos H.,73,69,1.900,1.900,36361,69,73,7,...,0,0,1.900,1.900,9,0,0.000,9,0.500000,0
36304,Del Potro J.M.,Anderson K.,9,8,1.500,2.620,36304,9,8,1,...,0,1,1.500,2.620,6,3,-1.120,6,0.635922,3
